In [ ]:
#!/usr/bin/env python3
"""Head-to-head comparison of two CSV files.

Reports whether they are identical and, if not, exactly how they differ:
file existence, column names/order, row count, and individual cell values.

Usage:
    python compare_csvs.py X.csv Y.csv
    python compare_csvs.py X.csv Y.csv --ignore-row-order
    python compare_csvs.py X.csv Y.csv --ignore-row-order --ignore-col-order

Exit code is 0 if the files are identical, 1 if they differ (handy in scripts/CI).
"""

import argparse
import os
import sys

import pandas as pd


def fmt(v):
    """Render a cell value cleanly (unwrap numpy scalars to native Python)."""
    if hasattr(v, "item"):
        try:
            v = v.item()
        except Exception:
            pass
    return repr(v)


def load_csv(path):
    if not os.path.exists(path):
        return None, f"File not found: {path}"
    try:
        return pd.read_csv(path), None
    except Exception as e:
        return None, f"Could not parse {path}: {e}"


def compare(path_x, path_y, ignore_row_order=False, ignore_col_order=False,
            max_report=20):
    df_x, err_x = load_csv(path_x)
    if err_x:
        print(err_x)
        return False
    df_y, err_y = load_csv(path_y)
    if err_y:
        print(err_y)
        return False

    same = True
    cols_x, cols_y = list(df_x.columns), list(df_y.columns)

    # --- Column names / order ---
    if set(cols_x) != set(cols_y):
        same = False
        only_x = [c for c in cols_x if c not in set(cols_y)]
        only_y = [c for c in cols_y if c not in set(cols_x)]
        print("[x] Column names differ")
        if only_x:
            print(f"      only in X: {only_x}")
        if only_y:
            print(f"      only in Y: {only_y}")
    elif cols_x != cols_y:
        if ignore_col_order:
            print("[-] Column order differs (ignored)")
        else:
            same = False
            print("[x] Column order differs")
            print(f"      X: {cols_x}")
            print(f"      Y: {cols_y}")
    else:
        print("[ok] Columns match")

    # Align column order if requested and the sets are equal
    if ignore_col_order and set(cols_x) == set(cols_y):
        df_y = df_y[cols_x]

    # --- Row count ---
    if df_x.shape[0] != df_y.shape[0]:
        same = False
        print(f"[x] Row count differs: X has {df_x.shape[0]}, Y has {df_y.shape[0]}")
    else:
        print(f"[ok] Row count matches ({df_x.shape[0]} rows)")

    # If columns don't match, a value comparison isn't meaningful
    if set(cols_x) != set(cols_y):
        print("\nResult: DIFFERENT (columns don't match)")
        return False

    # --- Optionally compare rows as an unordered set ---
    if ignore_row_order:
        df_x = df_x.sort_values(by=cols_x).reset_index(drop=True)
        df_y = df_y.sort_values(by=cols_x).reset_index(drop=True)

    # --- Cell values (only when shapes line up) ---
    if df_x.shape == df_y.shape:
        # NaN == NaN should count as equal
        unequal = (df_x != df_y) & ~(df_x.isna() & df_y.isna())
        n_diff = int(unequal.values.sum())
        if n_diff == 0:
            print("[ok] All cell values match")
        else:
            same = False
            print(f"[x] {n_diff} cell value(s) differ:")
            count = 0
            for col in df_x.columns:
                for idx in unequal.index[unequal[col]]:
                    print(f"      row {idx}, column '{col}': "
                          f"X={fmt(df_x.at[idx, col])}  Y={fmt(df_y.at[idx, col])}")
                    count += 1
                    if count >= max_report:
                        break
                if count >= max_report:
                    if n_diff - count > 0:
                        print(f"      ... and {n_diff - count} more")
                    break

    print("\nResult:", "IDENTICAL" if same else "DIFFERENT")
    return same
